# Ciudad de domicilios multiagente con aprendizaje por refuerzo

**Proyecto de Simulación de Sistemas**

Simulamos una ciudad donde **habitantes** con hambre piden comida, **restaurantes** la cocinan y **repartidores** la llevan.
El repartidor aprende con **Q-learning** qué pedidos aceptar y el habitante aprende con un **bandit ε-greedy** qué restaurante le conviene.

**Resultado principal.** Q-learning aprende: su recompensa supera a la de todas las heurísticas y entrega más rápido.
Pero en cancelaciones solo empata con las políticas aleatoria y "más cercano", y la regla "atender primero el pedido más viejo" es la mejor para la ciudad.
La sección 6 explica por qué, y es uno de los hallazgos centrales del proyecto.

> Este notebook corre versiones **reducidas** de los experimentos para que tarde pocos minutos (1 semilla, pocos episodios).
> Los resultados completos (5 semillas, 300 episodios de entrenamiento) están en las tablas de cada sección y en `resultados/` del repositorio.
> Los números de las celdas pueden variar un poco respecto a esas tablas.

## 0. Preparación

En Colab, esta celda descarga el repositorio e instala Mesa (NumPy, Pandas, Matplotlib y NetworkX ya vienen instalados).
Si el repositorio es privado, el `git clone` falla: hazlo público en GitHub o usa la celda opcional de abajo para subir el `.zip`.

In [ ]:
import os, sys

REPO = "https://github.com/CarlZeichenfolge/simulacion-domicilios-rl.git"
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    if not os.path.exists("simulacion-domicilios-rl"):
        !git clone -q {REPO}
    %cd simulacion-domicilios-rl
    !pip install -q "mesa>=3.0,<4" networkx pyyaml
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # corriendo local desde la carpeta notebooks/

sys.path.insert(0, os.getcwd())
print("Carpeta del proyecto:", os.getcwd())

In [ ]:
# OPCIONAL (solo si el repo es privado): sube el .zip del repositorio y descomprímelo.
# from google.colab import files
# subido = files.upload()                 # elige simulacion-domicilios-rl.zip
# !unzip -q -o simulacion-domicilios-rl.zip
# %cd simulacion-domicilios-rl
# !pip install -q "mesa>=3.0,<4" networkx pyyaml
# import sys, os; sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from experimentos.correr import correr_corrida
from experimentos.graficas import COLOR, SERIES, estilo
from sim.agentes import EstadoRepartidor
from sim.config import cargar_config, fusionar
from sim.metricas import resumen_episodio
from sim.model import CiudadModel

estilo()
BASE = cargar_config(cambios={"metricas": {"recolectar_cada": 0}})
print("Mesa listo. Repartidores por defecto:", BASE["repartidores"]["cantidad"],
      "| habitantes:", BASE["habitantes"]["cantidad"])

## 1. El entorno

La ciudad es una cuadrícula de 10 × 10 manzanas convertida en grafo de NetworkX: 121 intersecciones y 220 calles de 400 m.
Un tick es un minuto y un episodio es un día (1440 ticks).

| Elemento | Cómo funciona | Por qué así |
| --- | --- | --- |
| Calles | Arista con `longitud`, `tiempo_base` (1.33 min a 18 km/h) y `factor_trancon` | El tráfico es solo un peso que cambia, y A* trabaja sobre calles reales |
| Edificios | 40 casas, 8 oficinas, 1 universidad y 3 restaurantes | El mapa tiene su propia semilla: es el escenario fijo de cada experimento |
| Restaurantes | Pizza (12 min, \$25.000, 4 fogones), Hamburguesas (20 min, \$32.000, 3), Sushi (30 min, \$45.000, 2) | Rápido-barato frente a lento-caro obliga a elegir |
| Lluvia | Cadena de Markov por hora: empieza con p = 0.15, para con p = 0.35. Velocidad al 70 % | El clima cambia por horas, no por minutos |
| Trancones | En hora pico (7-9 y 17-19) el 30 % de las calles multiplica su tiempo ×1.5 a ×3 | Un trancón dura toda la franja; los repartidores replanifican |
| Pedido | CREADO → EN_PREPARACION → LISTO → EN_CAMINO → ENTREGADO; se cancela a los 60 min | Las transiciones inválidas lanzan error en vez de dejar un dato raro |

El mapa a las 8:10 de la mañana (hora pico, en rojo las calles con trancón):

In [ ]:
ESTILO_EDIFICIO = {"casa": ("#c9a27e", 45), "oficina": ("#7d93b2", 80), "universidad": ("#9a74c4", 140)}
COLOR_REPARTIDOR = {EstadoRepartidor.LIBRE: "#898781",
                    EstadoRepartidor.YENDO_A_RESTAURANTE: "#eda100",
                    EstadoRepartidor.YENDO_A_CLIENTE: "#2a78d6"}


def dibujar_ciudad(m, ax):
    # Solo LEE el modelo (igual que las vistas 2D y 3D del proyecto).
    ax.clear()
    for u, v, d in m.ciudad.grafo.edges(data=True):
        trancada = d["factor_trancon"] > 1
        ax.plot([u[0], v[0]], [u[1], v[1]], color="#e34948" if trancada else "#d9d8d2",
                lw=3 if trancada else 2, zorder=1)
    for tipo, (color, tam) in ESTILO_EDIFICIO.items():
        nodos = m.ciudad.nodos_de_tipo(tipo)
        ax.scatter([n[0] for n in nodos], [n[1] for n in nodos], marker="s", s=tam, color=color, zorder=2)
    for nombre, r in m.restaurantes.items():
        ax.scatter(*r.nodo, marker="s", s=320, color=COLOR[nombre], zorder=3, edgecolor="white", linewidth=2)
        ax.annotate(nombre, r.nodo, xytext=(0, 13), textcoords="offset points", ha="center", fontsize=9, weight="bold")
    activos = [p for p in m.pedidos_activos if not p.terminado]
    if activos:
        ax.scatter([p.destino[0] for p in activos], [p.destino[1] for p in activos],
                   marker="v", s=60, color="#e87ba4", zorder=4, label="pedido esperando")
    for rep in m.repartidores:
        x, y = rep.posicion_continua()
        ax.scatter(x, y, s=160, color=COLOR_REPARTIDOR[rep.estado], edgecolor="white", linewidth=2, zorder=5)
    clima = "lluvia" if m.eventos.lloviendo else "seco"
    trafico = "hora pico" if m.eventos.trancon_activo else "tráfico normal"
    ax.set_title(f"{m.reloj.texto()}  ·  {clima}  ·  {trafico}  ·  "
                 f"entregados {m.conteo['entregados']}  cancelados {m.conteo['cancelados']}", fontsize=10)
    ax.set_xlim(-0.6, m.ciudad.tamano + 0.6); ax.set_ylim(-0.6, m.ciudad.tamano + 0.6)
    ax.set_aspect("equal"); ax.axis("off")


modelo = CiudadModel(fusionar(BASE, {"repartidores": {"politica": "cercano"}}), semilla=3)
for _ in range(8 * 60 + 10):
    modelo.step()
fig, ax = plt.subplots(figsize=(7, 7))
dibujar_ciudad(modelo, ax)
plt.show()

### Propiedades del entorno

| Propiedad | Justificación |
| --- | --- |
| Multiagente | 30 habitantes, 3 restaurantes y 3 repartidores actúan a la vez; los repartidores compiten por los mismos pedidos |
| Estocástico | Lluvia, trancones, tiempos de cocina, precios y propinas son aleatorios (con semilla) |
| Dinámico | El mundo cambia aunque un agente no actúe: los pedidos envejecen y se cancelan, la lluvia empieza sola |
| Parcialmente observable | El repartidor no ve pedidos futuros ni el tiempo real de cocina; el habitante no ve la cola del restaurante |
| Discreto en el tiempo | Un tick = un minuto; las posiciones avanzan de forma continua dentro de cada calle |
| Secuencial | Aceptar un pedido lejano hoy deja al repartidor lejos del siguiente |

## 2. Agentes

Todos repiten el ciclo **percibir → decidir → actuar** en cada tick. La decisión se delega a una *política* intercambiable,
así que comparar políticas es cambiar una línea del YAML.

| Agente | Tipo | Por qué |
| --- | --- | --- |
| Habitante | Basado en utilidad | Compara cocinar, salir y pedir con gusto − peso_precio × precio − peso_tiempo × minutos |
| Restaurante | Basado en modelo | Decide según su estado interno (cola y fogones ocupados), no solo según lo que llega ahora |
| Repartidor | Basado en metas, con aprendizaje | Su meta es entregar; planea la ruta con A* y aprende con Q-learning qué pedido aceptar |

### Tablas PEAS

| | Habitante | Restaurante | Repartidor |
| --- | --- | --- | --- |
| **Medida de desempeño** | Satisfacción de cada comida; pocas cancelaciones | Pedidos listos a tiempo; cola corta | Ganancia neta (pago − costo por minuto); pocas cancelaciones |
| **Entorno** | Casa, oficina o universidad; menú de 3 restaurantes; clima | Su cocina con capacidad limitada; la demanda | Red de calles con trancones y lluvia; pedidos sin dueño |
| **Actuadores** | Cocinar, salir a comer, pedir a un restaurante | Empezar a cocinar, marcar listo | Aceptar un pedido, esperar, moverse, recoger, entregar |
| **Sensores** | Hambre, energía, dinero, actividad, lluvia, precios del menú | Su cola, sus fogones, la hora | Su posición, pedidos disponibles (distancia, edad, pago, si está listo), hora, lluvia |

## 3. Un día en la ciudad

Corremos un día completo con 3 repartidores ("más cercano") y miramos cómo se acumulan los pedidos.
Las franjas azules son las horas con lluvia.

In [ ]:
cfg_dia = fusionar(BASE, {"repartidores": {"politica": "cercano"}, "metricas": {"recolectar_cada": 10}})
dia = CiudadModel(cfg_dia, semilla=3)
dia.correr()
serie = dia.datacollector.get_model_vars_dataframe()
serie["hora"] = serie["tick"] / 60

fig, ax = plt.subplots(figsize=(10, 3.6))
for i in range(len(serie) - 1):
    if serie["lluvia"].iloc[i]:
        ax.axvspan(serie["hora"].iloc[i], serie["hora"].iloc[i + 1], color=SERIES[0], alpha=0.08, lw=0)
ax.plot(serie["hora"], serie["entregados"], color=SERIES[0], label="entregados")
ax.plot(serie["hora"], serie["cancelados"], color=SERIES[7], label="cancelados")
ax.set_xlabel("Hora del día"); ax.set_ylabel("Pedidos acumulados"); ax.set_xticks(range(0, 25, 3))
ax.legend(loc="upper left"); ax.set_title("Pedidos acumulados en un día (semilla 3)", loc="left")
plt.show()
pd.Series(resumen_episodio(dia)).loc[["pedidos_creados", "entregados", "cancelados", "tasa_cancelacion",
                                      "tiempo_entrega_promedio", "satisfaccion_promedio"]].round(3)

### La hora del almuerzo, animada

De 12:00 a 15:00, un cuadro cada 3 minutos. Amarillo = va al restaurante, azul = va al cliente, gris = libre; los triángulos rosados son pedidos esperando.

In [ ]:
from matplotlib import animation
plt.rcParams["animation.embed_limit"] = 40

anim_modelo = CiudadModel(fusionar(BASE, {"repartidores": {"politica": "cercano"}}), semilla=5)
for _ in range(12 * 60):
    anim_modelo.step()

fig, ax = plt.subplots(figsize=(6, 6), dpi=70)

def cuadro(_):
    for _ in range(3):
        anim_modelo.step()
    dibujar_ciudad(anim_modelo, ax)

anim = animation.FuncAnimation(fig, cuadro, frames=60, interval=150)
plt.close(fig)
HTML(anim.to_jshtml())

## 4. Aprendizaje

### 4.1 Repartidor: Q-learning tabular (SMDP)

| Elemento | Definición |
| --- | --- |
| Estado (contexto) | Franja horaria (6) × lluvia (2) × pedidos disponibles (0, 1-2, 3-5, 6+) = 48 contextos |
| Acción | Tipo de pedido que acepta: distancia al restaurante (4) × distancia de entrega (3) × edad (4) × ¿listo? (2) = 96, más **esperar** |
| Recompensa | Pago − 80 \$/min × minutos; −5.000 si se le cancela; −4.000 por rechazar (esperar habiendo pedidos). En miles |
| Exploración | ε-greedy; ε de 1.0 a 0.05, ×0.99 por episodio |

Aceptar un pedido dura 20-40 minutos y esperar dura 3, así que el futuro se descuenta por los $k$ minutos que duró la acción:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma^{k} \max_{a'} Q(s',a') - Q(s,a) \right], \quad \alpha = 0.1,\ \gamma = 0.97 \text{ por minuto}$$

Tabular y no una red: son 48 × 97 celdas, se entrena en minutos y cada celda se puede explicar.
Una sola tabla para todos los repartidores: son idénticos, así que juntar su experiencia acelera el aprendizaje.

### 4.2 Habitante: bandit ε-greedy

Cada restaurante es un brazo. $Q(r)$ estima la satisfacción con los tiempos y precios que el habitante **vivió**, no los prometidos:

$$Q(r) \leftarrow Q(r) + \alpha \left[ \text{satisfacción} - Q(r) \right], \qquad \text{satisfacción} = \text{gusto}_r - w_p \cdot \text{precio} - w_t \cdot \text{espera}$$

Cancelación = −5, ε = 0.1, α = 0.2 constante (los tiempos cambian con lluvia y trancones).
Bandit y no Q-learning porque escoger restaurante no cambia el estado futuro del habitante.

### 4.3 Entrenamiento

Entrenamos Q-learning 150 días para ver la curva (el experimento completo usa 300 días × 5 semillas). Tarda 1 a 2 minutos.
Aquí ε baja más rápido (×0.98 por día) para que termine explotando lo aprendido.

In [ ]:
import time

cfg_q = fusionar(BASE, {"repartidores": {"politica": "qlearning", "qlearning": {"decaimiento_epsilon": 0.98}}})
inicio = time.time()
filas_ent, _, politica_q = correr_corrida(cfg_q, semilla=1, episodios_entrenamiento=150, episodios_evaluacion=0)
print(f"Entrenado en {time.time() - inicio:.0f} s · epsilon final {politica_q.epsilon:.3f}")

curva = pd.DataFrame(filas_ent)
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(curva["episodio"], curva["recompensa_total"], color=SERIES[0], alpha=0.25, lw=1)
ax.plot(curva["episodio"], curva["recompensa_total"].rolling(15, min_periods=1).mean(), color=SERIES[0], lw=2.5,
        label="Q-learning (media móvil 15 días)")
ax.set_xlabel("Día de entrenamiento"); ax.set_ylabel("Recompensa total del día")
ax.set_title("Curva de aprendizaje del repartidor", loc="left"); ax.legend(loc="lower right")
plt.show()

## 5. Experimentos

**Metodología completa** (la de las tablas): 5 semillas; Q-learning se entrena 300 días; luego todas las variantes se evalúan
30 días con **las mismas semillas** (comparación pareada), sin explorar ni aprender. Valores = media ± desviación entre semillas.

**En este notebook:** 1 semilla y pocos días por variante, para que corra rápido.

In [ ]:
def evaluar(cambios, dias=8, politica=None, semilla=7):
    # Promedio de `dias` episodios de evaluación con semillas comunes a todas las variantes.
    filas, _, _ = correr_corrida(fusionar(BASE, cambios), semilla, 0, dias, politica=politica)
    return pd.DataFrame(filas).mean(numeric_only=True)

METRICAS = ["tasa_cancelacion", "tiempo_entrega_promedio", "ganancia_neta_por_repartidor", "satisfaccion_promedio"]

### 5.1 Políticas del repartidor

Resultados completos (5 semillas):

| Política | Cancelación | Tiempo de entrega (min) | Ganancia por repartidor (miles \$) | Satisfacción |
| --- | --- | --- | --- | --- |
| Urgente (el más viejo) | **9.1 % ± 1.0** | 36.1 ± 0.3 | **81.8 ± 0.7** | **1.68 ± 0.09** |
| Q-learning | 13.0 % ± 0.8 | 34.1 ± 0.2 | 80.9 ± 0.8 | 1.46 ± 0.10 |
| Más cercano | 14.0 % ± 0.9 | 35.1 ± 0.3 | 76.8 ± 1.8 | 1.34 ± 0.06 |
| Aleatoria | 14.1 % ± 1.2 | 34.3 ± 0.2 | 78.7 ± 0.9 | 1.37 ± 0.12 |
| Q-learning sin penalización por rechazo | 20.6 % ± 2.2 | **33.7 ± 0.3** | 78.5 ± 2.6 | 0.88 ± 0.20 |

"Urgente" es la mejor para la ciudad. Q-learning entrega más rápido y gana más que "cercano" y "aleatoria",
pero su ventaja en cancelación sobre ellas (1 punto) es del tamaño de la desviación entre semillas: no es concluyente.
En la versión rápida de abajo, con otros días, puede salir incluso un poco por encima de ellas.

Versión rápida. Para Q-learning usamos la tabla entrenada 300 días (`modelos/tabla_q_300dias.npz`), no la de 150 días de arriba:

In [ ]:
from sim.politicas import crear_politica_repartidor

politica_300 = crear_politica_repartidor(cfg_q["repartidores"])
politica_300.cargar("modelos/tabla_q_300dias.npz")

res = {}
for nombre in ["aleatoria", "cercano", "urgente"]:
    res[nombre] = evaluar({"repartidores": {"politica": nombre}})
res["qlearning"] = evaluar({"repartidores": {"politica": "qlearning"}}, politica=politica_300)
tabla1 = pd.DataFrame(res).T[METRICAS]
tabla1["ganancia_neta_por_repartidor"] /= 1000

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.4))
for eje, col, titulo in [(ejes[0], "tasa_cancelacion", "Tasa de cancelación"),
                         (ejes[1], "tiempo_entrega_promedio", "Tiempo de entrega (min)")]:
    eje.bar(tabla1.index, tabla1[col], color=[COLOR[n] for n in tabla1.index], edgecolor="white", linewidth=2)
    eje.set_title(titulo, loc="left")
ejes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout(); plt.show()
tabla1.round(3)

### 5.2 Cantidad de repartidores

Completos: con 1, 3 y 5 repartidores la cancelación es 68.4 %, 14.0 % y 3.1 %; el tiempo de entrega 43.2, 35.1 y 31.9 min;
pero cada repartidor gana 124, 77 y 48 mil pesos. Con uno solo se crean 125 pedidos al día (la gente vuelve a pedir tras cada cancelación).

In [ ]:
tabla2 = pd.DataFrame({n: evaluar({"repartidores": {"politica": "cercano", "cantidad": n}}, dias=5)
                       for n in [1, 3, 5]}).T
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.2))
ejes[0].plot(tabla2.index, tabla2["tasa_cancelacion"], marker="o", color=SERIES[0])
ejes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0)); ejes[0].set_title("Tasa de cancelación", loc="left")
ejes[1].plot(tabla2.index, tabla2["tiempo_entrega_promedio"], marker="o", color=SERIES[0]); ejes[1].set_title("Tiempo de entrega (min)", loc="left")
for e in ejes: e.set_xticks([1, 3, 5]); e.set_xlabel("Repartidores")
plt.tight_layout(); plt.show()
tabla2[["pedidos_creados", "tasa_cancelacion", "tiempo_entrega_promedio", "satisfaccion_promedio"]].round(3)

### 5.3 Lluvia

Completos: sin lluvia, estocástica y todo el día → pedidos 74.7 / 81.6 / 112.6 por día; salidas a comer 11.2 / 9.4 / 0.8;
cancelación 7.7 % / 14.0 % / 32.4 %. La lluvia no multiplica la demanda desde afuera: castiga la utilidad de salir a comer,
y el aumento de domicilios sale de la decisión de cada habitante.

In [ ]:
tabla3 = pd.DataFrame({modo: evaluar({"repartidores": {"politica": "cercano"}, "eventos": {"lluvia": {"modo": modo}}}, dias=5)
                       for modo in ["nunca", "estocastica", "siempre"]}).T
tabla3[["pedidos_creados", "comidas_salir", "tasa_cancelacion", "tiempo_entrega_promedio"]].round(3)

### 5.4 Preferencias de los habitantes

Completos (14 días, 5 semillas): los que aprenden suben Pizza de 27 % a 41 % de sus pedidos (es la más rápida y casi no se cancela);
el grupo de control (α = 0, nunca actualiza lo que cree) apenas pasa de 26 % a 32 %. Aprender baja la cancelación de 15.7 % a 8.1 %.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for eje, (titulo, alpha) in zip(ejes, [("Aprenden (α = 0.2)", 0.2), ("No aprenden (α = 0)", 0.0)]):
    cfg = fusionar(BASE, {"tiempo": {"dias_por_episodio": 14}, "repartidores": {"politica": "cercano"},
                          "habitantes": {"aprendizaje": {"alpha": alpha}}})
    _, comidas, _ = correr_corrida(cfg, 1, 0, 1, guardar_comidas=True)
    pedidos = pd.DataFrame(comidas).query("tipo == 'pedir'")
    fraccion = pd.crosstab(pedidos["dia"] + 1, pedidos["restaurante"], normalize="index")
    for r in ["Pizza", "Hamburguesas", "Sushi"]:
        eje.plot(fraccion.index, fraccion[r], marker="o", ms=4, color=COLOR[r], label=r,
                 lw=2.5 if r == "Pizza" else 1.5)
    eje.set_title(titulo, loc="left"); eje.set_xlabel("Día")
ejes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0)); ejes[0].legend()
plt.tight_layout(); plt.show()

## 6. Decisiones de diseño que salieron de los experimentos

| Qué vimos | Qué hicimos | Por qué |
| --- | --- | --- |
| "Más cercano" cancelaba más que "aleatoria" | Agregamos la heurística "urgente" (el pedido más viejo) | Ir siempre por lo cercano deja vencer los pedidos lejanos |
| Con acciones "cercano / mejor pagado / esperar", Q-learning esperaba casi siempre (28 % de cancelación) | La acción pasó a ser el tipo de pedido | Con reglas fijas como acciones no distinguía un pedido salvable de uno perdido |
| Sin castigo por rechazar, el repartidor escogía solo pedidos cómodos (21 %) | Penalización de 4.000 por rechazar | Maximizar la ganancia propia perjudicaba al sistema: incentivos mal alineados |
| Una recompensa "cooperativa" dio 37 % de cancelación | Se descartó | La multa se acumula por minuto; esperar (3 min) sumaba menos que entregar (30 min) |
| Casi nadie salía a comer, así que la lluvia no tenía efecto real | Cada habitante tiene su propio gusto por salir | Sin salidas que quitar, la lluvia no podía empujar hacia el domicilio |
| Q-learning no le gana a "urgente" | Probamos un pago que sube con la espera y la señal "¿es el más viejo?" (12-15 % frente a 8 %) | Ver abajo |

**Por qué Q-learning no supera a "urgente".** La ventaja de atender primero el pedido más viejo es **del sistema**: reduce las cancelaciones
de toda la ciudad. Pero la recompensa de cada repartidor es **individual**, y un agente que optimiza su propia ganancia no tiene por qué
descubrir la regla que mejor sirve a todos. Por eso en las plataformas reales la asignación la hace un despachador central.

## 7. Conclusiones

1. Los dos aprendizajes funcionan: la recompensa del repartidor sube de −13 a 225 en 300 días, y los habitantes que aprenden bajan la cancelación de 15.7 % a 8.1 %.
2. Q-learning optimiza bien su propio objetivo (recompensa y tiempo de entrega), pero en cancelaciones solo empata con aleatoria y más cercano, y una regla simple del dominio (FIFO) sigue siendo la mejor para el sistema.
3. El diseño de la recompensa pesó más que el algoritmo: el mismo Q-learning pasa de 13 % a 21 % de cancelación solo con quitar el castigo por rechazar.
4. Los efectos del entorno son grandes: 1/3/5 repartidores → 68 % / 14 % / 3 % de cancelación; lluvia todo el día → de 8 % a 32 %.

**Trabajo futuro:** un despachador central con recompensa del sistema; aproximación de funciones para estados continuos;
un predictor de demanda por hora para los restaurantes.

---
Repositorio: `simulacion-domicilios-rl` · las vistas 2D (Pygame) y 3D (Ursina) se corren en local con `python main.py --vista 2d` / `--vista 3d`.